# Data Loading, Cleaning, and Feature Engineering

## Introduction: The Modeling Question

In Project 1 we explored descriptive statistics to understand the Mexico City housing market — distributions, correlations, and grouped summaries. Now we cross into new territory: **supervised machine learning**. The goal is no longer just to *describe* data — it is to *predict* outcomes from data.

**Our core research question:**

> Can we accurately predict apartment prices in Buenos Aires based on characteristics like size, type, and location?

Answering that question is a three-stage process, all of which this lesson covers:

1. **Clean, structured data** — raw data is rarely machine-learning-ready straight from the source.
2. **Thoughtful feature engineering** — transforming raw columns into mathematical inputs a model can actually use.
3. **A proper train-test split** — ensuring our model is evaluated on data it has never seen during training.

### The Supervised Machine Learning Framework

In supervised learning, we provide a model with **labeled examples** — in our case, apartment listings where we already know the sale price — and ask it to learn the relationship between the listing's characteristics and its price. Once the model is trained, it can estimate prices for new listings it has never seen before.

This is called "supervised" because every training example comes with the correct answer (the actual sale price). The model adjusts its internal parameters to minimize the gap between its predictions and those known answers — essentially learning from its own mistakes.

The workflow divides into two distinct, non-overlapping phases:

| Phase | Data used | What happens |
|-------|-----------|--------------|
| **Training** | `X_train`, `y_train` | Model learns the feature-to-price mapping by minimizing prediction error |
| **Evaluation** | `X_test`, `y_test` | We measure how accurately it predicts prices it has **never seen** |

> 💡 **Why keep test data completely separate?**
>
> If we measured accuracy on the same data the model trained on, we would get an unrealistically optimistic result — the model would have effectively "memorized" the correct answers. The test set acts as the final exam: problems the model has never practiced on. Only test-set performance tells us whether the model has genuinely *learned* the relationship, or merely *memorized* specific examples.

This distinction — **learning** vs. **memorizing** — is the central tension of all of machine learning. We will return to it repeatedly throughout this project under the names *generalization* (good learning) and *overfitting* (too much memorization).

### What This Lesson Covers

By the end of this lesson, you will be able to:

1. Use `glob` to locate multiple CSV files matching a naming pattern and load them into a single DataFrame
2. Filter data to a homogeneous market segment using `.loc[]` with lambda functions and `.query()`
3. Remove outliers systematically using quantile-based bounds that adapt to any data distribution
4. Extract numeric and categorical features from raw string columns
5. Diagnose missing-value patterns using bar charts and the `missingno` matrix
6. Detect and resolve multicollinearity between numeric features using a correlation heatmap
7. Understand why one-hot encoding is needed for categorical variables, why `category_encoders` is preferred over scikit-learn's `OneHotEncoder`, and how to apply it
8. Split data into training and test sets with scikit-learn's `train_test_split`

---

## 1. Import and Merge Multiple CSV Files

The `data/` folder contains five Buenos Aires housing CSV files — one per year of apartment listings. Each file has the same 16-column structure, so we can safely stack them into a single DataFrame.

### 1.1 Why `glob` Instead of Hard-Coding File Paths?

When a directory contains multiple files that follow a predictable naming pattern, `glob` lets you express that pattern as a **wildcard string** and receive a list of all matching file paths automatically.

```python
from glob import glob
datasets = glob("./data/buenos-aires-real-estate-*.csv")
# Returns: ["./data/buenos-aires-real-estate-1.csv",
#           "./data/buenos-aires-real-estate-2.csv", ...]
```

The `*` wildcard matches any sequence of characters. One line of code finds all five CSVs without ever naming them individually.

**Why this matters beyond convenience:**

Hard-coding a list of filenames creates **brittle code** — the moment a filename changes or a sixth year of data is added, your script breaks silently or requires manual editing. `glob` is the robust alternative: it finds whatever files match the pattern at runtime, making the pipeline **self-updating** as new data arrives.

> 📌 **Pattern thinking:** The ability to describe a *set of things* by their shared characteristics (the glob pattern) rather than enumerating each one individually is a recurring pattern in data science — you will see it again in regular expressions, SQL `LIKE` queries, and DataFrame filtering. The underlying skill is the same: express a rule, let the system find the matches.

**The project arc:**

This project has four lessons. Each builds directly on the one before:

- **Lesson 1 (this lesson):** Prepare the data — load, clean, encode, and split
- **Lesson 2:** Fit a baseline and a linear regression model; measure performance with MAE, RMSE, and R²
- **Lesson 3:** Improve the model with regularization (Ridge and Lasso); learn to control complexity
- **Lesson 4:** Compare all models fairly on identical test data; select the best one for production

Every data preparation decision made here — which rows to keep, which columns to drop, how to encode `neighborhood`, which `random_state` to use — propagates forward into all three subsequent lessons. Getting this right matters.

**Code 2.1.1.1**

In [ ]:
import pandas as pd
import numpy as np
from glob import glob

datasets = glob("./data/buenos-aires-real-estate-*.csv")
datasets

### 1.2 Reading Multiple Files with a `for` Loop

Now that `datasets` holds the list of file paths, we need to read each one with `pd.read_csv` and collect the resulting DataFrames before stacking them.

The three-step pattern is:
1. Start with an empty list: `files = []`
2. Loop through each path — read the CSV, append the DataFrame to `files`
3. After the loop, stack all DataFrames at once with `pd.concat`

**Why concatenate after the loop, not inside it?**

Appending to a list is cheap — Python just adds a pointer. Concatenating DataFrames is expensive — pandas creates a new DataFrame every time. If you called `pd.concat` inside the loop, you would be creating and copying progressively larger DataFrames five times. Collecting into a list first, then calling `pd.concat` once at the end, performs the same work in a fraction of the memory and time.

> 🧠 **The key habit:** Separate the *collecting* phase from the *combining* phase. This pattern appears whenever you are aggregating results from multiple sources — files, API calls, database queries.

**Code Task 2.1.1.1**

In [ ]:
files = []

for x in datasets:
    data = pd.read_csv(...)  # <--- Read the CSV file at path x
    files.append(data)

len(files)

> **What `x` is doing**
>
> `x` is a **loop variable** — it takes on each successive value from `datasets` during the loop. If `datasets = ["./data/file1.csv", "./data/file2.csv", "./data/file3.csv"]`:
>
> - **Iteration 1:** `x = "./data/file1.csv"` → reads file1 → appends DataFrame to `files`
> - **Iteration 2:** `x = "./data/file2.csv"` → reads file2 → appends DataFrame to `files`
> - **Iteration 3:** `x = "./data/file3.csv"` → reads file3 → appends DataFrame to `files`
>
> The name `x` is arbitrary — `path`, `filename`, or `csv_file` would work equally well. It simply holds one item at a time so the loop body can work with it.

`ignore_index=True` in `pd.concat` resets the row indices so they run from 0 to N-1 across the combined DataFrame. Without it, each sub-DataFrame would keep its original indices — you would see repeated index values (0, 1, 2, … 0, 1, 2, …) which causes subtle bugs when slicing or merging later.

**Code Task 2.1.1.2**

In [ ]:
df = pd.concat(..., ignore_index=True)  # pd.concat(files, ignore_index=True)
df.head()

### 1.3 From a Loop to a Function: List Comprehension

The loop above works, but we can make it more concise and reusable by converting it into a **function** and replacing the `for` loop with **list comprehension**.

**List comprehension** is a compact Python syntax that builds a list in a single expression:

```python
# for-loop version — explicit, step by step
files = []
for x in datasets:
    data = pd.read_csv(x)
    files.append(data)

# List comprehension — same result, one line
files = [pd.read_csv(x) for x in datasets]
```

Reading the comprehension left-to-right: "Build a list containing `pd.read_csv(x)` for each `x` in `datasets`." The structure is always `[expression for item in iterable]`.

> 💡 **When to prefer list comprehension:** Use it when you are building a list with a single, clear transformation — one input, one output. If the loop body involves conditional logic, multiple operations, or side effects, a regular `for` loop is usually easier to read. Clarity beats brevity.

We will wrap this into a function called `merged_files` so that any future notebook can call it with a single line and get back the fully merged DataFrame.

**Code 2.1.1.3**

In [ ]:
def merged_files(data):
    """Merge multiple CSV files matching a pattern into a single DataFrame."""
    return pd.concat([pd.read_csv(file) for file in glob(data)], ignore_index=True)

# Usage
merged_files("./data/buenos-aires-real-estate-*.csv").info()

**Now it is time to answer MCQ 2.1.1.1.**

---

## 2. Data Cleaning

Raw data is rarely machine-learning-ready straight from the source. Our Buenos Aires dataset, now merged into a single DataFrame of over 43,000 rows and 16 columns, has several structural issues that must be addressed before we can train a reliable model.

**The problems we need to solve:**

| Problem | Effect on model if ignored |
|---------|---------------------------|
| **Market heterogeneity** — listings span multiple property types and cities | Model learns inconsistent patterns; predictions are mediocre everywhere |
| **Extreme outliers** — unusual surface area values | Regression coefficients bend toward extreme cases, reducing accuracy for typical properties |
| **Missing values** — some columns are >50% empty | Filling with mean/median invents data; better to drop and avoid noise |
| **Redundant features** — highly correlated columns | Model cannot isolate individual feature effects; coefficients become unstable |
| **String-encoded numbers** — coordinates stored as `"-34.60,-58.38"` | Cannot be used in matrix multiplication until split and cast to float |
| **Data leakage** — price-derived columns in the feature set | Model "cheats" by using the answer; fails completely on new listings |

We will tackle each issue in sequence, then bundle all cleaning steps into a single `clean_files()` master function.

### 2.1 Filter to Capital Federal Only

> **Why Filter to Capital Federal?**
>
> The raw dataset spans **Capital Federal** (the autonomous city of Buenos Aires) and **Greater Buenos Aires (GBA)** — two fundamentally different real estate markets that should not be modeled together.
>
> - **Capital Federal:** Dense urban core where walkability, transit proximity, building amenities, and neighborhood prestige drive high prices per square meter. The key features are size and location *within* the city.
> - **Greater Buenos Aires:** Diverse suburban and semi-rural zones where lot size, commute distance, and proximity to commercial centers matter more. A different set of features drives prices.
>
> Mixing them produces a **heterogeneous dataset** — a model trained on both would be mediocre at predicting in either market, because the pricing logic differs. One linear equation cannot simultaneously capture urban Palermo dynamics and suburban La Matanza dynamics.
>
> **Additional filters:**
> - **Apartments only** — houses and commercial spaces follow different pricing mechanics (lot size, zoning, business revenue potential)
> - **Price < $400,000** — caps the range to typical residential buyers, excluding luxury outliers whose prices are driven by factors our features do not capture

**Two tools for row filtering: `.loc[]` and `.query()`**

Both accomplish the same thing — keeping rows that satisfy a condition and discarding the rest. The difference is syntax:

```python
# .loc[] with lambda — explicit, works with complex conditions
df.loc[lambda x: x["property_type"] == "apartment"]

# .query() — cleaner string syntax, easier to read for simple conditions
df.query('property_type == "apartment"')
```

Inside a method chain, `.loc[lambda x:]` is preferred because it references the DataFrame at its current state in the chain rather than needing a separate variable. `.query()` is excellent for readable ad-hoc filtering but cannot use lambda patterns.

> 💡 **Rule of thumb:** Use `.query()` for simple, readable filters when writing interactive exploration code. Use `.loc[lambda x:]` inside pipeline functions because it works with any DataFrame variable name and integrates cleanly into method chains.

**A note on the price cap:**

Capping at $400,000 removes properties whose prices are driven by luxury factors our features do not capture — exclusive building amenities, historic architecture, penthouses with private pools. These are real signals, but they require different features (building age, floor number, doorman presence) that our dataset does not consistently record. Including them would introduce noise that our simple linear model cannot explain, degrading predictions for the vast majority of typical residential properties.

**Code 2.1.2.1**

In [ ]:
(
    df
    .loc[lambda x: x["place_with_parent_names"].str.contains("Capital Federal")]
    .query('property_type == "apartment"')
    .query('price_aprox_usd < 400_000')
    .head() # <--- remember to delete this after copying to form the function
)

Now wrap the filtering logic into a reusable function called `filter_df`. The function accepts a DataFrame called `data` and returns the filtered version.

> 📌 **Before running:** Remove `.head()` from the chain — `head()` is for inspection only and should not be part of pipeline functions. The function should return all filtered rows, not just the first five.

**Code Task 2.1.2.1**

In [ ]:
def filter_df(data):
    return (...)

**Now it is time to answer MCQ 2.1.2.1.**

### 2.2 Handling Outliers

Even within Capital Federal apartments under $400K, the surface area column (`surface_covered_in_m2`) still contains extreme values that would distort the model's learned coefficients.

**Why outliers harm linear regression — the intuition**

Linear regression minimizes the **sum of squared errors** (SSE). Because errors are *squared*, a prediction that is 100 units off contributes 10,000 to the SSE, while a prediction that is 10 units off contributes only 100. A single extreme outlier can dominate the entire optimization — the model bends its coefficients to better accommodate that one extreme point, at the cost of accuracy for the thousands of typical properties.

> **Visual intuition:** Imagine drawing a line of best fit through a scatter plot. If one point sits at ten times the normal range, the best-fit line tilts noticeably toward it — making it a worse fit for the 99% of points in the normal range. Linear regression is *not* robust to outliers by default; this is a fundamental limitation of ordinary least squares.

**What the extreme values represent in our data**

- **< 10th percentile** (very small surface area): Could be storage units, misclassified commercial spaces, or data-entry errors. These are not apartments in any practical sense.
- **> 90th percentile** (very large surface area): Luxury penthouses or large commercial-residential hybrids with different pricing mechanisms — not typical residential apartments.

Including either group would force the model to try to learn pricing patterns from properties that do not belong in the target market.

**The quantile-based approach**

```python
# Keep only properties between the 10th and 90th percentile of surface area
data.loc[lambda x: x["surface_covered_in_m2"].between(
    x["surface_covered_in_m2"].quantile(0.1),   # 10th percentile lower bound
    x["surface_covered_in_m2"].quantile(0.9)    # 90th percentile upper bound
)]
```

**Why quantiles instead of fixed cutoffs?**

Using `.quantile(0.1)` and `.quantile(0.9)` rather than hard-coded numbers like `20` and `150` makes the filter **adaptive**: it adjusts automatically if the data distribution changes (new data added, different market segment). A fixed cutoff that works today might be wrong after the dataset is updated — quantile-based filtering is robust to such shifts.

> ⚠️ **Caveat:** Removing the top and bottom 10% is a deliberate choice — we are building a model for *typical* apartments. If the goal were to model the luxury market, we would use a different filter. Always match your data preparation choices to your modeling objectives.

**Code Task 2.1.2.2**

In [ ]:
def outliers_df(data):
    return (
        data
        .loc[lambda x: x[...].between(x[...].quantile(...), x[...].quantile(...))]  # <--- x["surface_covered_in_m2"].between(x["surface_covered_in_m2"].quantile(0.1), x["surface_covered_in_m2"].quantile(0.9))
    )

# Test the function
check_outliers = outliers_df(df)
check_outliers.info()

**Now it is time to answer MCQ 2.1.2.2.**

### 2.3 Extracting Features from String Columns

Our dataset contains two columns that store **multiple pieces of information bundled into a single string**. A machine learning model cannot work with strings directly — we need to extract the numeric and categorical information encoded inside them.

**What is buried in each column?**

| Column | Raw value example | Hidden information | Target features |
|--------|-------------------|--------------------|-----------------|
| `"lat-lon"` | `"-34.6037,-58.3816"` | Latitude and longitude as a comma-separated pair | `lat` (float), `lon` (float) |
| `"place_with_parent_names"` | `"Argentina|Buenos Aires|Capital Federal|Palermo|..."` | Hierarchical location path, pipe-separated | `neighborhood` (string, segment index 3) |

**Why latitude and longitude matter for price prediction**

Geographic coordinates are among the strongest predictors of real estate prices. Properties closer to the city center, major transit hubs, or desirable amenities command higher prices. By extracting `lat` and `lon` as separate numeric features, we let the model learn north-south and east-west price gradients independently.

**Why neighborhood as a separate feature?**

Coordinates capture continuous geographic variation, but `neighborhood` captures *discrete categorical identity* — the local amenity profile, prestige, and community character of a named area. These are complementary signals: two apartments 500 meters apart might be in different neighborhoods with meaningfully different price expectations.

**The extraction technique: `.str.split(expand=True)`**

```python
# Split "lat-lon" on comma, take piece [0] as latitude
lat = lambda x: x["lat-lon"].str.split(",", expand=True)[0].astype(float)

# Split "lat-lon" on comma, take piece [1] as longitude
lon = lambda x: x["lat-lon"].str.split(",", expand=True)[1].astype(float)

# Split "place_with_parent_names" on pipe, take piece [3] as neighborhood
neighborhood = lambda x: x["place_with_parent_names"].str.split("|", expand=True)[3]
```

`expand=True` makes `.str.split()` return a DataFrame of columns (one per piece) instead of a Series of lists — allowing direct indexing with `[0]`, `[1]`, `[3]`.

> ⚠️ **`.astype(float)` is required** for `lat` and `lon`. After splitting, every piece is still a string — even though it looks like a number. `.astype(float)` converts `"-34.6037"` into the actual floating-point number `-34.6037` that arithmetic and modeling require.

**Code Task 2.1.2.3: Complete the `modify_cols` function**

Fill in the three `...` placeholders with the correct column names (as quoted strings). Review the table above for the mapping: which column holds the comma-separated coordinates? Which holds the pipe-separated location hierarchy?

**Code Task 2.1.2.3**

In [ ]:
def modify_cols(data):
    return (
        data
        .assign(
            lat=lambda x: x[...].str.split(",", expand=True)[0].astype(float),      # <--- "lat-lon"
            lon=lambda x: x[...].str.split(",", expand=True)[1].astype(float),      # <--- "lat-lon"
            neighborhood=lambda x: x[...].str.split("|", expand=True)[3]            # <--- "place_with_parent_names"
        )
    )

check_cols = modify_cols(df)
check_cols.info()

### 2.4 Missing Values Analysis

After filtering, outlier removal, and feature extraction, we need to understand where our data has gaps. Missing values in machine learning are not just an inconvenience — they require a deliberate decision for each affected column.

**The three options for handling missing values:**

| Option | When to use it | Risk |
|--------|---------------|------|
| **Imputation** (fill with mean, median, mode) | Missingness rate is low (<15%) and the column is important | Invented data adds noise; can distort model training if overused |
| **Model-based imputation** | Feature is valuable and pattern of missingness is predictable | Complex to implement correctly; can leak information |
| **Drop the column** | Missingness rate is high (>50%) | Loses potentially useful information, but avoids inventing data for the majority of rows |

**Our approach:** Visualize first, decide second. Two complementary plots reveal both the *magnitude* of missingness (how many values are missing per column?) and the *pattern* (do rows tend to be missing multiple columns together?).

> 🔍 **Why does the pattern matter?**
>
> If `currency`, `price_aprox_local_currency`, and `price_aprox_usd` are always missing together, they share the same source — likely listings that did not record price information at all. Imputing one while keeping the others would create a logically inconsistent record. Understanding *co-occurrence* of missingness guides better decisions.

**Code 2.1.2.4**: Missing Values Visualization

In [ ]:
import matplotlib.pyplot as plt

# Percentage of missing values by column
nan_columns_percentage = (df.isna().sum(axis=0) * 100 / len(df)).sort_values()

fig, ax = plt.subplots()

# Horizontal bar plot
ax.barh(y=nan_columns_percentage.index, width=nan_columns_percentage)
ax.set_title('Percentage of Missing Values by Column')
ax.set_xlabel('Percentage (%)')
ax.set_ylabel('Column')
plt.show()

The bar chart shows the **percentage of missing values per column**. Key observations:

- `expenses` and `floor` are missing more than 50% of their values
- `rooms` is missing around 40%
- `price_aprox_usd` — our target — has some missing values but is mostly complete

> ⚠️ **The 50% threshold is not arbitrary.** When a column is >50% missing, any imputation strategy (mean, median, mode) is inventing values for the majority of rows. The "filled" values would introduce more artificial noise than real signal. The principled choice is to drop these columns and rely on the information that is genuinely present.

Now let's look at the **missing value matrix** from the `missingno` library — a more granular view that reveals not just *how much* is missing, but *which rows* are missing values across multiple columns simultaneously.

**Code 2.1.2.5**: Missing Values Matrix

In [ ]:
! pip install missingno -q
import missingno as msno

# Plot missing values with color royalblue
fig, ax = plt.subplots()
msno.matrix(df, color=(0.3, 0.4, 0.8), ax=ax)
plt.show()

> **Reading the `missingno` Matrix**
>
> The matrix uses color to encode data presence: **blue = value present**, **white = value missing**. Reading across a row tells you which columns a specific property record is missing. Reading down a column tells you how consistently that feature was recorded across all listings.
>
> **What we observe:**
> - Columns like `operation`, `property_type`, and `currency` are **solid blue** — fully complete (expected: we filtered to one value of each, so every remaining row has a value)
> - `currency`, `price_aprox_local_currency`, and `price_aprox_usd` tend to be **missing together** — these co-occur in rows where no price information was recorded at all
> - `expenses` has the most white space, followed by `floor` and `rooms`
>
> **Decision:** Keep `price_aprox_usd` (our target variable) and drop all other price-related and high-missingness columns. The pattern of co-occurrence with other price columns confirms that rows missing `price_aprox_usd` are simply unpriced listings — we cannot recover them.

### 2.5 Multicollinearity Analysis

After resolving missing values, we face a different problem: some columns that *are* present may be measuring nearly the same thing, making them redundant and potentially harmful to the model.

**What is multicollinearity?**

Multicollinearity occurs when two or more features in the dataset are **highly correlated** with each other — they move together almost perfectly, conveying the same underlying information.

**Why does this matter for linear regression?**

The linear regression equation assigns a coefficient $\beta_i$ to each feature $x_i$, representing how much the predicted price changes per unit increase in $x_i$ *holding all other features constant*. When two features are highly correlated, the phrase "holding all other features constant" becomes nearly impossible — you cannot change `surface_covered_in_m2` while keeping `surface_total_in_m2` fixed because they move together.

The consequence: both coefficients become **unstable and unreliable**. Small perturbations in the data cause large swings in the learned values. The model's predictions may still be reasonable, but the individual coefficients become uninterpretable — you cannot say which feature is actually driving price.

**The canonical example in our dataset:**

`surface_total_in_m2` (total area including balconies, terraces, parking) and `surface_covered_in_m2` (indoor living area only) are nearly perfectly correlated — correlation > 0.9. Keeping both gives the model two near-identical signals and produces unstable, uninterpretable coefficients.

**Detection strategy:** A Seaborn correlation heatmap lets us visually scan for pairs with correlation above ~0.8. Pairs that are dark red or dark blue (strong positive or negative correlation) are candidates for removal.

**Reading the heatmap:**

- **Dark red (near +1.0):** Two features move together almost perfectly — as one increases, the other increases proportionally. Classic example in our data: total area and covered area.
- **Dark blue (near -1.0):** Two features move in opposite directions almost perfectly — equally problematic for coefficient stability.
- **Light colors (near 0):** Little to no linear relationship — these feature pairs can safely coexist in the model.

**What to do when you find multicollinear features:**

Two main options:
1. **Drop one:** Keep the more interpretable or more complete feature. Drop the other. This is what we will do for `surface_total_in_m2` — we keep `surface_covered_in_m2` because indoor living area is the more standard way buyers evaluate apartment size.
2. **Create a composite feature:** Combine both into a single new column (e.g., ratio of covered to total area as a measure of how "enclosed" the apartment is). This preserves information from both columns. We will not use this approach here, but it is a valid alternative when both columns encode genuinely distinct information you want to retain.

**Code Task 2.1.2.4**

In [ ]:
import seaborn as sns

# Calculate correlation matrix for numeric columns
corr_matrix = df.select_dtypes(include='number').corr()  # <--- .corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Matrix')
plt.show()

### 2.6 Dropping Unnecessary Columns

Based on the missing value analysis and the multicollinearity heatmap, we now have a principled list of columns to remove. Each drop has a specific reason — this is not arbitrary data reduction.

> **Principled Column Removal — Full Rationale**
>
> | Category | Columns | Why they must go |
> |----------|---------|-----------------|
> | **High missingness (>50%)** | `expenses`, `floor`, `rooms` | Imputing >50% of values creates more fictional data than real; any model trained on them would learn noise |
> | **Data leakage** | `price`, `price_aprox_local_currency`, `price_usd_per_m2`, `price_per_m2` | These columns are **computed from the target price**. If left in, the model learns to predict price by reading price — achieving perfect training accuracy but being completely useless on new listings where these derived columns do not exist |
> | **Multicollinearity** | `surface_total_in_m2` | Correlation >0.9 with `surface_covered_in_m2` — keeping both makes both coefficients unstable and uninterpretable |
> | **Unique identifiers** | `properati_url` | Every listing has a unique URL — the model would memorize specific URLs rather than learning generalizable price patterns. A 0% test-set accuracy waiting to happen |
> | **Already extracted** | `lat-lon`, `place_with_parent_names` | We extracted `lat`, `lon`, and `neighborhood` from these — the original string columns are now redundant |
> | **Single-value columns** | `operation`, `property_type`, `currency` | We filtered to one value of each (`"sell"`, `"apartment"`, `"USD"`). A column with zero variance provides zero information to the model |

> ❗️ **Data leakage deserves special emphasis.** It is the most dangerous category because it produces results that look excellent during development — the model seems to perform perfectly — but fail catastrophically when deployed. A real estate platform cannot include `price_usd_per_m2` in a new listing before it has been sold. If the model depends on that column, it is not predicting the price; it is reading it. Always ask: *"Would this feature be available at prediction time?"*

**Code 2.1.2.6**

In [ ]:
drop_cols = [
    "lat-lon", "place_with_parent_names",
    "floor", "expenses", "rooms", "price",
    "price_aprox_local_currency", "price_usd_per_m2",
    "price_per_m2", "operation", "property_type", "currency",
    "properati_url", "surface_total_in_m2"
]

### 2.7 The Master Cleaning Function

We now have five discrete cleaning functions:
- `merged_files()` — loads all CSVs and stacks them into one DataFrame
- `filter_df()` — keeps only Capital Federal apartments under $400K
- `outliers_df()` — removes the bottom and top 10% by surface area
- `modify_cols()` — extracts lat, lon, and neighborhood from string columns
- A drop step — removes all unnecessary columns

Running these five operations in the correct sequence every time we start a new notebook would be error-prone. One skipped step would silently corrupt the data used for modeling. Instead, we bundle them into a single master function `clean_files()` using the pandas **`.pipe()`** method.

> **The `.pipe()` Method — Chaining Custom Functions**
>
> `.pipe(func)` passes the current DataFrame as the first argument to `func` and returns the result. This lets you chain custom functions in the same readable dot-notation as built-in pandas methods:
>
> ```python
> # Without .pipe() — nested, reads right-to-left, hard to follow
> result = drop_cols(modify_cols(outliers_df(filter_df(merged_files(path)))))
>
> # With .pipe() — reads left-to-right like a recipe
> result = (merged_files(path)
>           .pipe(filter_df)
>           .pipe(outliers_df)
>           .pipe(modify_cols)
>           .pipe(drop_cols))
> ```
>
> The `.pipe()` chain reads as a **recipe**: merge, then filter, then remove outliers, then extract features, then drop columns. Each step receives the output of the previous step as its input.

**Why a single master function matters across this project:**

In Lessons 2, 3, and 4, we will call `clean_files()` exactly once at the top of each notebook. This guarantees that every lesson starts from *identical* cleaned data — a prerequisite for fair model comparison. If Lesson 3 used slightly different filtering than Lesson 2, any performance difference we observe could be due to the data difference rather than the model improvement. The master function eliminates this confound.

> 📌 **Software engineering principle:** Don't Repeat Yourself (DRY). When the same sequence of operations appears in multiple places, wrap it in a function. This also means that if you discover a bug in one cleaning step, you fix it in one place and all lessons benefit automatically.

**Code 2.1.2.7**

In [ ]:
def clean_files(file_path):
    """Final function to complete the cleaning process"""
    drop_cols = [
        "lat-lon", "place_with_parent_names",
        "floor", "expenses", "rooms", "price",
        "price_aprox_local_currency", "price_usd_per_m2",
        "price_per_m2", "operation", "property_type", "currency",
        "properati_url", "surface_total_in_m2"
    ]

    return (
        merged_files(file_path)      # <--- Initial DataFrame
        .pipe(filter_df)             # <--- Filter to Capital Federal apartments
        .pipe(outliers_df)           # <--- Remove outliers
        .pipe(modify_cols)           # <--- Create new columns
        .drop(columns=drop_cols)     # <--- Drop unnecessary columns
        .dropna()                    # <--- Drop rows with missing values
    )

clean_df = clean_files("./data/buenos-aires-real-estate-*.csv")
clean_df.info()
clean_df

**Verifying the pipeline:** After calling `clean_files()`, the resulting DataFrame should have:
- Fewer rows than the raw merged dataset (outliers removed, market filtered)
- Fewer columns (high-missingness and redundant columns dropped)
- Three new columns: `lat`, `lon`, `neighborhood`
- No more `lat-lon` or `place_with_parent_names` (replaced by the extracted versions)

> 📊 **What to look for in the output:**
>
> Compare `df.info()` before and after `clean_files()`. The row count drop tells you how many properties fell outside Capital Federal, outside the price cap, or in the outlier quantiles. The column count drop tells you how many redundant or high-missingness columns were removed. If either number seems unexpectedly large or small, trace back through the individual functions to find which step is removing too much or too little.

This diagnostic habit — comparing shapes before and after transformations — is a fundamental data quality check. Develop it as a reflex: every time you apply a non-trivial transformation, check that the output dimensions make sense.

**Now it is time to answer MCQ 2.1.2.4.**

---

## 3. Split Data into Features and Target

Before we can encode categorical variables or train any model, we must separate the DataFrame into two distinct components that scikit-learn expects: the **feature matrix** and the **target vector**.

### The X/y Convention

This separation mirrors the mathematical notation of supervised learning. Every scikit-learn model expects:

- **`X`** — the feature matrix: a 2D array where rows are observations (individual apartments) and columns are features (measurable characteristics used to predict price). We use uppercase `X` because it represents a 2-dimensional matrix.
- **`y`** — the target vector: a 1D array containing the value we want to predict — `price_aprox_usd`. We use lowercase `y` because it is a 1-dimensional vector (one number per row).

> 💡 **Why does this convention matter?** Scikit-learn's API is consistent across all models: `model.fit(X_train, y_train)` and `model.predict(X_test)`. When you learn this convention once, it applies to every scikit-learn estimator — linear regression, Ridge, Lasso, Random Forest, XGBoost. The `X`/`y` split is the universal entry point to the entire scikit-learn ecosystem.

**What goes in `X` and what goes in `y`?**

| Component | Content | Shape |
|-----------|---------|-------|
| `y` | `price_aprox_usd` — the value we predict | `(n_rows,)` — one price per listing |
| `X` | Every other column — `surface_covered_in_m2`, `lat`, `lon`, `neighborhood` | `(n_rows, n_features)` — one row per listing, one column per feature |

**A critical order-of-operations note:**

We perform the X/y split *before* one-hot encoding and *before* the train-test split. Why?

1. One-hot encoding must be fit *only* on training data (so test data cannot influence the encoding). That requires knowing which rows are training rows first.
2. The train-test split must happen on the complete feature set so that both `X_train` and `X_test` have identical columns.

The correct sequence: clean data → X/y split → train-test split → encoding (inside the pipeline, fit on training only).

**Code Task 2.1.3.1**

In [ ]:
target = "price_aprox_usd"
features = [cols for cols in clean_df.columns if cols not in target]

y = clean_df[...]  # <--- clean_df[target]
X = clean_df[...]  # <--- clean_df[features]

print(f"y shape: {y.shape}")
print(f"X shape: {X.shape}")

> **Understanding the Code**
>
> ```python
> target = "price_aprox_usd"
> ```
> Stores the target column name as a string — what we want to predict.
>
> ```python
> features = [cols for cols in clean_df.columns if cols not in target]
> ```
> List comprehension: for each column name in the DataFrame, keep it *only if* it is not the target variable. This produces the list of all feature columns, automatically excluding the target.
>
> **Why `not in target` and not `!= target`?**
>
> Because `target` is a string (`"price_aprox_usd"`), `not in target` checks whether the column name is a substring of that string — which works here, but is technically imprecise. In more general code, you would write `cols != target` or `cols not in [target]`. The result is the same for our dataset.

---

## 4. Our Modeling Goal: The Linear Regression Equation

Everything we have done — merging, filtering, removing outliers, feature extraction, handling missing values, resolving multicollinearity — has been preparation for one purpose: fitting a **linear regression model** that predicts apartment prices.

### The Equation

Linear regression assumes that the target can be expressed as a weighted sum of the input features plus a constant:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_3 + \beta_4 x_4$$

**Decoding each symbol:**

| Symbol | Name | In our context |
|--------|------|---------------|
| $\hat{y}$ | Predicted price | The model's output — the estimated sale price for a given apartment |
| $\beta_0$ | Intercept | The predicted price when all features equal zero — a mathematical baseline |
| $\beta_1$ | Coefficient for $x_1$ | How much predicted price changes per additional square meter |
| $\beta_2$ | Coefficient for $x_2$ | How much predicted price changes per degree of latitude (north-south shift) |
| $\beta_3$ | Coefficient for $x_3$ | How much predicted price changes per degree of longitude (east-west shift) |
| $\beta_4$ | Coefficient for $x_4$ | How much predicted price changes for being in a specific neighborhood (after encoding) |
| $x_1$ | `surface_covered_in_m2` | Apartment size in square meters |
| $x_2$ | `lat` | Latitude coordinate |
| $x_3$ | `lon` | Longitude coordinate |
| $x_4$ | `neighborhood` | Categorical indicator — one-hot encoded into multiple binary columns |

**In plain language:** "Take the apartment's size in square meters and multiply it by the size coefficient. Take its latitude and multiply by the latitude coefficient. Take its longitude and multiply by the longitude coefficient. For its neighborhood, add the corresponding neighborhood coefficient. Sum all of these with the intercept. That sum is the predicted price."

### What the Model Is Learning

During training, scikit-learn finds the values of $\beta_0, \beta_1, \beta_2, \beta_3, \beta_4$ that minimize the total prediction error across all training examples. This optimization is called **Ordinary Least Squares (OLS)** — it minimizes the sum of squared differences between predicted and actual prices:

$$\text{minimize} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

### Why Every Cleaning Decision Had a Consequence

Each data preparation step was not just housekeeping — it had a direct mathematical effect on whether the equation above produces reliable results:

- **Outliers removed:** Squared errors from extreme values dominate the OLS objective. One outlier with a price error of $200,000 contributes $4 \times 10^{10}$ to the sum — equivalent to 40,000 predictions that are $1,000 off. The model bends coefficients toward that outlier at the expense of typical cases.
- **Multicollinearity resolved:** When $x_1$ and $x_2$ are nearly identical, the matrix equation that OLS solves becomes nearly **singular** (not invertible). The coefficients $\beta_1$ and $\beta_2$ blow up to compensate for each other — large positive and large negative values that happen to cancel out for training data but generalize poorly.
- **Data leakage eliminated:** Columns like `price_usd_per_m2` are derived from the target. Keeping them gives the OLS equation a shortcut: it can achieve near-zero training error by learning $\hat{y} \approx x_{\text{price\_per\_m2}} \times \text{surface}$ — but this "model" cannot be used in production where sale prices are unknown.
- **Missing values handled:** OLS requires a complete matrix — no `NaN` values. Rows with missing values in key features are simply dropped from the calculation.

> 🎯 **The takeaway:** Data cleaning is not a preliminary chore — it is a mathematical prerequisite for the OLS equation to be solvable, stable, and generalizable.

In Lessons 2, 3, and 4, we will fit this equation, measure its accuracy, improve it with regularization, and select the best version.

---

## 5. Encoding Categorical Variables: One-Hot Encoding

Our linear regression equation operates on **numbers** — it uses multiplication and addition. But the `neighborhood` feature contains text: `"Palermo"`, `"Recoleta"`, `"San Telmo"`. Text cannot participate in the equation until it is converted to numeric form.

### Why Not Assign Integers?

The obvious fix — replace `"Palermo"` with `1`, `"Recoleta"` with `2`, `"San Telmo"` with `3` — is deeply problematic. The model would interpret these as a numeric scale:

- "Recoleta is twice as valuable as Palermo" (2 vs. 1)
- "San Telmo is 50% more valuable than Recoleta" (3 vs. 2)

These implications are arbitrary and false. Neighborhoods have no such ordering. The integer encoding imposes a structure that does not exist in reality.

### The Solution: One-Hot Encoding

One-hot encoding creates **one binary column per unique category**. Each new column answers a single yes/no question: "Is this property located in neighborhood X?"

**Before encoding:**

| `neighborhood` |
|----------------|
| Palermo        |
| Recoleta       |
| San Telmo      |
| Palermo        |

**After one-hot encoding:**

| `nbhd_Palermo` | `nbhd_Recoleta` | `nbhd_SanTelmo` |
|:--------------:|:---------------:|:---------------:|
| 1              | 0               | 0               |
| 0              | 1               | 0               |
| 0              | 0               | 1               |
| 1              | 0               | 0               |

No implied ordering. No misleading numeric scale. Each property has exactly one `1` and the rest `0`s — a clean, interpretable representation of categorical membership.

**The high-cardinality problem**

Buenos Aires has 50+ neighborhoods. Naively one-hot encoding all of them adds 50+ new columns and creates four compounding problems:

1. **Curse of dimensionality:** The model needs proportionally more data to reliably learn patterns with 50+ extra columns. Many neighborhoods may have only dozens of listings — too few to estimate a reliable coefficient.
2. **Overfitting risk:** With enough neighborhood-specific columns, the model can memorize micro-level location quirks rather than learning the general price signal. It performs well on training data but poorly on new listings.
3. **Sparse data:** Each row has exactly one `1` across 50+ neighborhood columns. Most of the data is zeros — computationally wasteful.
4. **Computational cost:** Larger feature matrices mean more memory, slower training, and slower iteration during development.

> 💡 **Why `category_encoders` instead of scikit-learn's `OneHotEncoder`?**
>
> Scikit-learn's `OneHotEncoder` generates column names like `x0_0`, `x0_1`, `x0_2` — anonymous indices. When you inspect which neighborhoods drive price most strongly, you cannot tell which column represents Palermo and which represents Caballito.
>
> **`category_encoders.OneHotEncoder` generates interpretable names:** `neighborhood_Palermo`, `neighborhood_Recoleta`, `neighborhood_SanTelmo`. Every coefficient maps directly to a readable neighborhood name.
>
> This matters enormously for three practical reasons:
> - **Debugging:** You can spot whether a neighborhood that should be important has a reasonable coefficient
> - **Stakeholder communication:** You can explain model results in human-readable terms
> - **Feature selection:** Lessons 3 and 4 will drop neighborhood columns with near-zero coefficients — you need to know which ones to drop
>
> We make this choice here — in the first lesson where one-hot encoding appears — so that all downstream steps inherit readable, debuggable feature names.

**Now it is time to answer MCQ 2.1.3.1.**

---

**Code 2.1.5.1**

In [ ]:
# Start with cleaned data, select relevant columns, and show first few rows
sample_df = (
    clean_df
    [['neighborhood', 'price_aprox_usd']]  # <--- Select only these two columns
    .head(5)  # <--- Get first 5 rows to see the original format
)

print("BEFORE One-Hot Encoding:")
print(sample_df)
print(f"\nUnique neighborhoods in sample: {sample_df['neighborhood'].unique()}")

Now let's apply one-hot encoding:

**Code Task 2.1.5.2**

In [ ]:
# Transform categorical column into multiple binary columns
encoded_df = (
    sample_df
    .assign(
        is_palermo = (...).astype(int),  # <--- Create binary column: 1 if Palermo, 0 otherwise
        is_caballito = (...).astype(int),  # <--- Create binary column: 1 if Caballito, 0 otherwise
        is_villa_luro = (...).astype(int),  # <--- Create binary column: 1 if Villa Luro, 0 otherwise
    )
    .drop(columns=['neighborhood'])  # <--- Remove the original text column (we no longer need it)
)

print("\nAFTER One-Hot Encoding:")
print(encoded_df)

**Inspecting the encoded DataFrame:**

After running the encoding, examine the column names and values:

```python
# How many columns did we add?
print("Columns before encoding:", sample_df.shape[1])
print("Columns after encoding:", encoded_df.shape[1])
print("New neighborhood columns:", encoded_df.shape[1] - sample_df.shape[1])

# What do the new columns look like?
neighborhood_cols = [c for c in encoded_df.columns if c.startswith("neighborhood_")]
print("\nNeighborhood columns created:", neighborhood_cols)
```

Each new column has a name like `neighborhood_Palermo` — a direct product of using `category_encoders`. If you had used scikit-learn's `OneHotEncoder`, those same columns would be named `x0_0`, `x0_1`, `x0_2` with no way to tell which neighborhood they represent without consulting a separate lookup table.

> 🔍 **The dummy variable trap:** One-hot encoding for $k$ categories produces $k$ binary columns, but a linear regression model only needs $k-1$ of them. The $k$-th neighborhood is implicitly encoded by all others being `0`. Including all $k$ introduces **perfect multicollinearity** (the sum of all neighborhood columns always equals 1), which makes the OLS matrix singular. Scikit-learn handles this automatically with `drop="first"` or similar settings — and `category_encoders` handles it too. In practice, you will see one fewer column than you expect, and that is correct behavior.

**What happened:**

- Each row now has `0`s and `1`s in place of neighborhood text
- A property in Palermo has `neighborhood_Palermo = 1` and all other neighborhood columns set to `0`
- The model can now use these binary columns in the linear regression equation:

$$\hat{y} = \beta_0 + \beta_1(\text{surface\_covered\_in\_m2}) + \beta_2(\text{neighborhood\_Palermo}) + \beta_3(\text{neighborhood\_Caballito}) + \ldots$$

- Each $\beta$ for a neighborhood column represents the **price premium or discount** for that neighborhood compared to the baseline (the neighborhood that was dropped to avoid the dummy variable trap)

> 🧠 **Reading a neighborhood coefficient:** If $\beta_{\text{Palermo}} = 15{,}000$, the model has learned that — all else equal — being in Palermo is associated with approximately $15,000 more than the baseline neighborhood. Named columns make this interpretation immediate. Anonymous columns like `x0_5` make it impossible without looking up a mapping table.

**In Practice (Lessons 3 & 4):**

Rather than manually creating encoded DataFrames, we will use `category_encoders.OneHotEncoder` inside a scikit-learn `Pipeline`. The pipeline handles encoding automatically and — critically — ensures the encoder is fit only on training data and then applied identically to test data. This prevents a subtle but serious form of data leakage: allowing test-set category frequencies to influence how categories are encoded.

---

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169844382", h="3298dbabb7", width=700, height=450) 

## 6. Train-Test Split in Scikit-Learn

> **Why a Train-Test Split?**
>
> A train-test split divides the dataset into two **non-overlapping** portions:
>
> - **Training set (80%):** Used to teach the model — scikit-learn sees these examples and adjusts the regression coefficients to minimize prediction error
> - **Test set (20%):** Held out completely — the model never sees these during training; we use them only to measure real-world performance
>
> Evaluating on training data would be like a professor grading students on the exact practice questions they used to study. Of course they score well — but that score tells you nothing about whether they actually understood the material. The test set is the "real exam": problems the model has never practiced on.
>
> **The key insight:** A model's training accuracy and its test accuracy are two very different numbers. A large gap between them is the signature of overfitting — the model memorized training examples rather than learning generalizable patterns.

### The 80/20 Rule — Convention, Not Law

The 80/20 ratio reflects a fundamental tension in every machine learning project:

- The model **learns** from training data — more training data means more patterns to discover, more stable coefficient estimates, better generalization
- The model is **evaluated** on test data — more test data means a more reliable, lower-variance performance estimate

80/20 balances these competing needs for most medium-sized datasets. The right split depends on your data volume:

| Dataset size | Typical split | Reasoning |
|-------------|---------------|-----------|
| Small (<1,000 rows) | 90/10 | Cannot afford to lose training data; every row matters |
| Medium (1,000–100,000 rows) | 80/20 | Classic default — our Buenos Aires dataset falls here |
| Large (100,000+ rows) | 95/5 or 99/1 | Even 1% of 100,000 rows gives 1,000 test examples — plenty for reliable evaluation |

### `random_state=42` — Reproducibility

`random_state` seeds the random number generator so the split is **identical every time** the cell is executed. Without it:

- Two students running the same notebook would get different splits
- Model comparisons between Lessons 2, 3, and 4 would be meaningless — each model might have trained on different properties
- Debugging would be nearly impossible — results would change between runs for no visible reason

The number `42` is a convention (a nod to *The Hitchhiker's Guide to the Galaxy*), but any integer produces an equally valid seed. The requirement is **consistency**: use the same `random_state` throughout the entire project so every model comparison is fair.

> 💡 **Critical implication for this project:** Lessons 2, 3, and 4 all call `clean_files()` and then `train_test_split` with `random_state=42`. This guarantees that the Linear Regression model in Lesson 2, the Ridge and Lasso models in Lesson 3, and the comparison table in Lesson 4 all train and evaluate on *identical* data splits. Any performance differences reflect the models, not the data.

**Code Task 2.1.6.1**

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., random_state=42  # <--- X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

**The four variables produced by `train_test_split`:**

| Variable | Contents | Used for |
|----------|----------|----------|
| `X_train` | Feature matrix — training rows | Input to `model.fit()` |
| `X_test` | Feature matrix — test rows | Input to `model.predict()` during evaluation |
| `y_train` | Target vector — training prices | Labels for `model.fit()` — the "correct answers" the model trains against |
| `y_test` | Target vector — test prices | Ground truth for evaluation — compared against the model's predictions |

> 📌 **A common mistake:** Never fit (train) the model on any test data. The strict separation is the entire point. Even accidentally using a single row from `X_test` during training contamiates the evaluation — the test score will be optimistically biased. In scikit-learn, this separation is enforced by your own discipline: `model.fit(X_train, y_train)`, full stop.

The `assert` statements in the next cell verify that your split variables exist and have the correct names — a lightweight but effective guard against typos that would cause confusing errors in later cells.

In [ ]:
assert 'X_train' in globals(), "❌ You must create a variable named 'X_train'."
assert 'X_test' in globals(), "❌ You must create a variable named 'X_test'."
assert 'y_train' in globals(), "❌ You must create a variable named 'y_train'."
assert 'y_test' in globals(), "❌ You must create a variable named 'y_test'."
assert len(X_train) + len(X_test) == len(X), "❌ Train and test sets don't add up to original size."
assert abs(len(X_test) / len(X) - 0.2) < 0.01, "❌ Test set should be approximately 20% of the data."
assert len(X_train) == len(y_train), "❌ X_train and y_train must have the same length."
assert len(X_test) == len(y_test), "❌ X_test and y_test must have the same length."
print("✅ Train-test split completed successfully!")

**Now it is time to answer MCQ 2.1.6.1.**

---

## 7. Generalization and Overfitting: Key Concepts

Before we move to model training, two foundational concepts deserve explicit attention. They underlie every modeling decision in this project and every project thereafter.

> **Generalization**
>
> **Generalization** is a model's ability to perform well on **new, unseen data** after training on a specific dataset. A model that generalizes well has learned the *underlying statistical relationships* between features and price — patterns that hold for any apartment in Buenos Aires, not just the specific ones in the training set.
>
> **The exam analogy:** Genuine understanding versus rote memorization. A student who truly understands calculus can solve any integral they have never seen. A student who memorized 200 specific practice problems can only answer those exact 200 problems. Good generalization is the machine learning equivalent of true understanding.
>
> **In our context:** A well-generalizing model learns "larger apartments in desirable neighborhoods with good transit access tend to cost more" — a pattern it can confidently apply to a new listing it has never seen. A poorly generalizing model might learn "apartment at Avenida Corrientes 1234, listed in September 2014, costs $287,000" — a memorized fact, not a transferable rule.

> **Overfitting**
>
> **Overfitting** occurs when a model learns the training data **too well** — including its noise, outliers, and random peculiarities — rather than capturing the true underlying relationship.
>
> An overfit model achieves very high (sometimes near-perfect) accuracy on training data but fails badly on test data, because it has memorized the specific examples rather than learned the general rule.
>
> **Common causes of overfitting:**
> - **Model too complex** for the amount of data — too many free parameters relative to training examples
> - **Irrelevant features** — unique identifiers like `properati_url` give the model something to memorize instead of learning
> - **Outliers not removed** — extreme cases pull coefficients away from the true relationship for typical cases
> - **Data leakage** — price-derived features let the model "read the answer" during training
>
> **The signature of overfitting in practice:** training accuracy ≫ test accuracy. A model with 95% training accuracy but 60% test accuracy has overfit severely — it memorized the training set rather than learning.

> **Preventing Overfitting — What We Have Already Done**
>
> Every data preparation step in this lesson was partly about preventing overfitting:
>
> - **Train-test split:** Evaluation on held-out data reveals overfitting that would be invisible on training data
> - **Feature selection:** Removing `properati_url` eliminates an obvious memorization target
> - **Outlier removal:** Extreme values cause large coefficient distortion — removing them makes the learned relationship more stable
> - **Data leakage prevention:** Removing price-derived columns forces the model to learn from genuine features rather than shortcuts
> - **Market homogeneity:** Filtering to Capital Federal apartments reduces the diversity the model must simultaneously handle
>
> In Lesson 3, we will introduce **regularization** — a more systematic technique for controlling overfitting by penalizing overly large coefficients. The concepts you internalize here — what overfitting looks like, why it happens, and how the train-test split detects it — are the prerequisites for understanding why regularization is necessary.

---

## Summary

In this lesson, you built the complete data preparation pipeline for Buenos Aires apartment price prediction. Every step served a specific purpose in enabling reliable modeling.

| Step | What we did | Why it matters for modeling |
|------|-------------|----------------------------|
| **`glob` + `pd.concat`** | Loaded 5 CSV files matching a pattern | All available data in one DataFrame without hard-coding filenames |
| **`filter_df`** | Capital Federal apartments < $400K | Homogeneous market → more learnable, consistent patterns |
| **`outliers_df`** | 10th–90th percentile surface area | Prevents extreme values from distorting OLS coefficient estimates |
| **`modify_cols`** | Extracted lat, lon, neighborhood from strings | Geographic signals in numeric/categorical form the model can use |
| **Missing value analysis** | Dropped columns with >50% missing | Avoids inventing data for the majority of rows |
| **Multicollinearity analysis** | Dropped `surface_total_in_m2` | Prevents unstable, uninterpretable coefficient pairs |
| **Drop redundant columns** | Removed leakage, identifiers, single-value cols | Ensures the model learns real patterns, not shortcuts |
| **`clean_files()` master function** | Bundled all steps via `.pipe()` | One-call pipeline used identically in every subsequent lesson |
| **X/y split** | Feature matrix vs. target vector | Required entry point to the scikit-learn API |
| **One-hot encoding** | `neighborhood` → interpretable binary columns | Converts categorical data to numbers without implying order |
| **Train-test split** | 80/20, `random_state=42` | Fair evaluation on unseen data; reproducible across all lessons |

**Key Takeaways:**

- **Data cleaning is not a preliminary chore** — every decision has a direct mathematical consequence for the model's coefficients, stability, and generalizability
- **`category_encoders` over scikit-learn's `OneHotEncoder`** because named columns (`neighborhood_Palermo`) are interpretable; anonymous indices (`x0_5`) are not
- **The train-test split is the foundation of honest model evaluation** — without it, training accuracy is a meaningless metric
- **Generalization is the goal; overfitting is the enemy** — all the cleaning steps we applied this lesson are forms of overfitting prevention
- Understanding *why* each decision was made is as important as the technical implementation — future lessons will build on this reasoning

---

## Discussion Questions

1. Why did we filter to Capital Federal apartments under $400,000? What would happen to a model trained on the full dataset spanning all Buenos Aires regions and property types?

2. What would happen if we did not remove the highly correlated price-related columns before modeling? Give a specific example of how `price_usd_per_m2` could let the model "cheat."

3. Why is one-hot encoding performed *before* the train-test split in our pipeline? What specific problem would arise if we encoded *after* the split?

4. The lesson used a 10th–90th percentile filter for outliers. What are the tradeoffs of using a tighter filter (e.g., 20th–80th percentile) or a looser one (e.g., 5th–95th percentile)?

5. Can you think of additional features we could engineer from the existing columns that might improve price prediction? What raw information in the dataset remains unexploited?